In [0]:
%run /Workspace/Users/bernabeglennmarkc@gmail.com/superstore/utilities/util_config

In [0]:
%run /Workspace/Users/bernabeglennmarkc@gmail.com/superstore/utilities/util_helpers

## 1. Read from Silver Orders

In [0]:
log("Reading from Silver orders ...")
df_silver_orders = spark.table(TBL_SILVER_ORDERS)

display(df_silver_orders.limit(10))

## 2. Extract Unique Ship Modes

In [0]:
df_ship_modes = df_silver_orders.select('ship_mode').distinct().dropna()
df_ship_modes.show()

## 3. Generate Surrogate Key

In [0]:
from pyspark.sql.window import Window

df_dim_shipment = df_ship_modes.withColumn(
    "shipment_id",
    F.dense_rank().over(Window.orderBy(F.col("ship_mode")))
).select(
    "shipment_id",
    "ship_mode"
)

log(f"Total shipment records: {df_dim_shipment.count():,}")

## 4. Write/Upsert to Gold

In [0]:
if spark.catalog.tableExists(TBL_GOLD_DIM_SHIPMENT):
  log(f"Upserting into {TBL_GOLD_DIM_SHIPMENT} ...")

  delta_table = DeltaTable.forName(spark, TBL_GOLD_DIM_SHIPMENT)
  (
      delta_table.alias("target")
      .merge(
          df_dim_shipment.alias("source"),
          "target.ship_mode = source.ship_mode"
      )
      .whenMatchedUpdateAll()
      .whenNotMatchedInsertAll()
      .execute()
  )

  log(f"✅ Done. Table {TBL_GOLD_DIM_SHIPMENT} upserted")

else:
  log(f"Creating {TBL_GOLD_DIM_SHIPMENT}")
  (
      df_dim_shipment.write
          .format("delta")
          .mode("overwrite")
          .saveAsTable(TBL_GOLD_DIM_SHIPMENT)
  )

  log(f"✅ Done. Table {TBL_GOLD_DIM_SHIPMENT} created")
